In [1]:
import sys, os, inspect
sys.path.insert(0, os.path.abspath(".."))

import torch
from losses import (
    ModelOutputs,
    adversarial_loss_discriminator, adversarial_loss_generator,
    cycle_consistency_loss, identity_loss,
    temporal_discriminator_loss, temporal_generator_loss,
    roi_cycle_consistency_loss,
)

torch.manual_seed(0)

## Dummy ModelOutputs (small shapes, for the image-space losses)

In [2]:
B, T, X, Y, Z = 2, 5, 8, 8, 8
C_feat = 16
Db, Hb, Wb = 2, 2, 2
A_glob, A_spat = 8, 4
patch = (2, 2, 2)

def rand(*shape):
    return torch.randn(*shape, requires_grad=True)

x_a = rand(B, T, X, Y, Z)
x_b = rand(B, T, X, Y, Z)

out = ModelOutputs(
    c_a=rand(B, C_feat, Db, Hb, Wb), c_b=rand(B, C_feat, Db, Hb, Wb),
    a_global=rand(B, A_glob), a_spatial=rand(B, A_spat, Db, Hb, Wb),
    a_global_b=rand(B, A_glob), a_spatial_b=rand(B, A_spat, Db, Hb, Wb),
    x_hat_b=rand(B, T, X, Y, Z), x_hat_a=rand(B, T, X, Y, Z),
    x_self_a=rand(B, T, X, Y, Z), x_self_b=rand(B, T, X, Y, Z),
    c_hat_b=rand(B, C_feat, Db, Hb, Wb), c_hat_a=rand(B, C_feat, Db, Hb, Wb),
    a_hat_global=rand(B, A_glob), a_hat_spatial=rand(B, A_spat, Db, Hb, Wb),
    x_cycle_a=rand(B, T, X, Y, Z), x_cycle_b=rand(B, T, X, Y, Z),
    score_real_b=rand(B, 1, *patch), score_fake_b=rand(B, 1, *patch),
    score_real_a=rand(B, 1, *patch), score_fake_a=rand(B, 1, *patch),
)
print("x_a", tuple(x_a.shape), " out.x_hat_b", tuple(out.x_hat_b.shape), " out.score_real_b", tuple(out.score_real_b.shape))

x_a (2, 5, 8, 8, 8)  out.x_hat_b (2, 5, 8, 8, 8)  out.score_real_b (2, 1, 2, 2, 2)


## 1. Adversarial loss (LSGAN) -- discriminator + generator

In [3]:
print(inspect.getsource(adversarial_loss_discriminator))
print(inspect.getsource(adversarial_loss_generator))

def adversarial_loss_discriminator(out: ModelOutputs,
                                    real_target: float = 1.0,
                                    fake_target: float = 0.0) -> Dict[str, Tensor]:
    """
    LSGAN discriminator loss with label smoothing.
    Real → real_target, fake → fake_target for both D_A and D_B.
    """
    real_b = torch.full_like(out.score_real_b, real_target)
    fake_b = torch.full_like(out.score_fake_b, fake_target)
    real_a = torch.full_like(out.score_real_a, real_target)
    fake_a = torch.full_like(out.score_fake_a, fake_target)

    L_D_B = (0.5 * F.mse_loss(out.score_real_b, real_b) +
             0.5 * F.mse_loss(out.score_fake_b, fake_b))

    L_D_A = (0.5 * F.mse_loss(out.score_real_a, real_a) +
             0.5 * F.mse_loss(out.score_fake_a, fake_a))

    return {"D_B": L_D_B, "D_A": L_D_A, "total": L_D_B + L_D_A}

def adversarial_loss_generator(out: ModelOutputs) -> Tensor:
    """
    LSGAN generator loss — works with single-scale tensor or 

In [4]:
d_losses = adversarial_loss_discriminator(out)
for k, v in d_losses.items():
    print(f"D {k:<6} {v.item():.4f}")

g_adv = adversarial_loss_generator(out)
print(f"G adv    {g_adv.item():.4f}")
g_adv.backward(retain_graph=True)
print("backward OK, grad reached score_fake_b:", out.score_fake_b.grad is not None)

D D_B    1.4212
D D_A    1.4374
D total  2.8586
G adv    2.8605
backward OK, grad reached score_fake_b: True


## 2. Cycle-consistency loss (L1)

In [5]:
print(inspect.getsource(cycle_consistency_loss))

def cycle_consistency_loss(out: ModelOutputs,
                            x_a: Tensor,
                            x_b: Tensor) -> Tensor:
    """
    L1 cycle loss. A→B→A should recover x_a; B→A→B should recover x_b.
    L1 over L2: preserves sharp temporal transitions in PSC-normalised data.
    """
    return F.l1_loss(out.x_cycle_a, x_a) + F.l1_loss(out.x_cycle_b, x_b)



In [6]:
L_cyc = cycle_consistency_loss(out, x_a, x_b)
print(f"L_cyc = {L_cyc.item():.4f}")

# sanity: identical cycle output -> input should give exactly 0
out_identity = ModelOutputs(**{**vars(out), "x_cycle_a": x_a, "x_cycle_b": x_b})
print(f"L_cyc (x_cycle == x) = {cycle_consistency_loss(out_identity, x_a, x_b).item():.6f}")

L_cyc = 2.2632
L_cyc (x_cycle == x) = 0.000000


## 3. Identity loss (L1)

In [7]:
print(inspect.getsource(identity_loss))

def identity_loss(out: ModelOutputs,
                  x_a: Tensor,
                  x_b: Tensor) -> Tensor:
    """
    L1 identity loss. Self-reconstruction should be a no-op.
    Prevents generators from changing input that does not need changing.
    """
    return F.l1_loss(out.x_self_a, x_a) + F.l1_loss(out.x_self_b, x_b)



In [8]:
L_idt = identity_loss(out, x_a, x_b)
print(f"L_idt = {L_idt.item():.4f}")

out_identity2 = ModelOutputs(**{**vars(out), "x_self_a": x_a, "x_self_b": x_b})
print(f"L_idt (x_self == x) = {identity_loss(out_identity2, x_a, x_b).item():.6f}")

L_idt = 2.2250
L_idt (x_self == x) = 0.000000


## 4. ROI-timeseries adversarial loss -- discriminator + generator

In [9]:
print(inspect.getsource(temporal_discriminator_loss))
print(inspect.getsource(temporal_generator_loss))

def temporal_discriminator_loss(
    real_output: Dict[str, Tensor],
    fake_output: Dict[str, Tensor],
    lambda_roi: float = 0.5,
) -> Dict[str, Tensor]:
    """
    LSGAN loss for MultiScaleROITemporalDiscriminator (models/roi_discriminator.py),
    combining its whole-brain "global" score with its per-ROI "roi" scores.

    Args:
        real_output: {"global": (B, 1), "roi": (B, n_rois)} — discriminator
                     output on real ROI timeseries
        fake_output: {"global": (B, 1), "roi": (B, n_rois)} — discriminator
                     output on corrected/generated ROI timeseries
        lambda_roi:  weight on the per-ROI term relative to the global term

    Returns:
        Dict with "total" (for .backward()) plus the individual components.
    """
    # Global real/fake losses
    loss_global_real = F.mse_loss(
        real_output["global"], torch.ones_like(real_output["global"])
    )
    loss_global_fake = F.mse_loss(
        fake_output["global"], torch.zeros_

In [10]:
n_rois, T_roi = 20, 5
real_output = {"global": rand(B, 1), "roi": rand(B, n_rois)}
fake_output = {"global": rand(B, 1), "roi": rand(B, n_rois)}

d_roi = temporal_discriminator_loss(real_output, fake_output, lambda_roi=0.5)
for k, v in d_roi.items():
    print(f"D_roi {k:<12} {v.item():.4f}")

g_roi = temporal_generator_loss(fake_output, lambda_roi=0.5)
for k, v in g_roi.items():
    print(f"G_roi {k:<12} {v.item():.4f}")
g_roi["total"].backward()
print("backward OK, grad reached fake_output['roi']:", fake_output["roi"].grad is not None)

D_roi total        1.5970
D_roi global       0.7930
D_roi roi          1.6081
D_roi global_real  1.3534
D_roi global_fake  0.2326
D_roi roi_real     2.1977
D_roi roi_fake     1.0186
G_roi total        1.1742
G_roi global       0.2691
G_roi roi          1.8103
backward OK, grad reached fake_output['roi']: True


## 5. ROI-timeseries cycle-consistency loss (L1)

In [11]:
print(inspect.getsource(roi_cycle_consistency_loss))

def roi_cycle_consistency_loss(
    input_roi_ts: Tensor,
    cycle_roi_ts: Tensor,
) -> Tensor:
    """
    L1 cycle-consistency loss on ROI mean-BOLD timeseries.

    Distinct from fc_loss, which compares a single forward hop's *FC
    (correlation) matrix* (x_a vs x_hat_b) — this compares the *raw*
    timeseries after the full round trip A->B->A (x_a vs x_cycle_a), the
    same relationship cycle_consistency_loss already enforces in voxel
    space (see losses.py #2), just computed in ROI-timeseries space
    instead. Should decrease as the cyclic reconstruction's ROI dynamics
    converge back to the real input's.

    Args:
        input_roi_ts: (B, n_rois, T) ROI timeseries of the real input (x_a)
        cycle_roi_ts: (B, n_rois, T) ROI timeseries of the cyclic
                      reconstruction (x_cycle_a, A->B->A)

    Returns:
        Scalar L1 loss.
    """
    assert input_roi_ts.shape == cycle_roi_ts.shape, (
        f"roi_cycle_consistency_loss: shape mismatch "
      

In [12]:
input_roi_ts = rand(B, n_rois, T_roi)
cycle_roi_ts = rand(B, n_rois, T_roi)

L_roi_cyc = roi_cycle_consistency_loss(input_roi_ts, cycle_roi_ts)
print(f"L_roi_cyc = {L_roi_cyc.item():.4f}")
print(f"L_roi_cyc (identical) = {roi_cycle_consistency_loss(input_roi_ts, input_roi_ts.clone()).item():.6f}")

L_roi_cyc = 1.0720
L_roi_cyc (identical) = 0.000000


## 6. Total loss -- generator update vs discriminator update

Matches `generator_loss()` in losses.py plus the ROI terms train.py adds on top,
and the D-side split (image D_A/D_B vs roi_disc -- separate optimizers, never summed).

In [13]:
# Weights as used in slurm/run_train_grade.sh
w_adv, w_cyc, w_idt = 1.0, 10.0, 5.0
w_roi_adv, w_roi_cycle = 1.0, 1.0

# --- Generator update: ONE total, all terms share opt_G ---
total_loss_G = (w_adv * g_adv + w_cyc * L_cyc + w_idt * L_idt
                + w_roi_adv * g_roi["total"] + w_roi_cycle * L_roi_cyc)

print("Generator update (opt_G, single backward):")
print(f"  adv       {w_adv} * {g_adv.item():.4f} = {(w_adv * g_adv).item():.4f}")
print(f"  cyc       {w_cyc} * {L_cyc.item():.4f} = {(w_cyc * L_cyc).item():.4f}")
print(f"  idt       {w_idt} * {L_idt.item():.4f} = {(w_idt * L_idt).item():.4f}")
print(f"  roi_adv   {w_roi_adv} * {g_roi['total'].item():.4f} = {(w_roi_adv * g_roi['total']).item():.4f}")
print(f"  roi_cycle {w_roi_cycle} * {L_roi_cyc.item():.4f} = {(w_roi_cycle * L_roi_cyc).item():.4f}")
print(f"  total_loss_G = {total_loss_G.item():.4f}")

# --- Discriminator update: TWO separate totals, TWO separate optimizers ---
D_total_image = d_losses["total"]        # opt_D.zero_grad(); .backward(); opt_D.step()
D_total_roi   = d_roi["total"]           # opt_D_roi.zero_grad(); .backward(); opt_D_roi.step()

print("\nDiscriminator update (two independent backward/step calls, NOT added together):")
print(f"  D_total_image (D_A + D_B) = {D_total_image.item():.4f}  -> opt_D")
print(f"  D_total_roi   (roi_disc)  = {D_total_roi.item():.4f}  -> opt_D_roi")

Generator update (opt_G, single backward):
  adv       1.0 * 2.8605 = 2.8605
  cyc       10.0 * 2.2632 = 22.6325
  idt       5.0 * 2.2250 = 11.1251
  roi_adv   1.0 * 1.1742 = 1.1742
  roi_cycle 1.0 * 1.0720 = 1.0720
  total_loss_G = 38.8643

Discriminator update (two independent backward/step calls, NOT added together):
  D_total_image (D_A + D_B) = 2.8586  -> opt_D
  D_total_roi   (roi_disc)  = 1.5970  -> opt_D_roi


## 7. Update procedure (train.py, per training step)

**Step 1 -- Generator update**
1. Freeze D_A/D_B/roi_disc: `requires_grad_(False)`. Unfreeze G/E_c/E_a: `requires_grad_(True)`.
2. Forward pass: `out = model(x_a, x_b, detach_fakes_for_D=False)` -- fakes (`x_hat_a`, `x_hat_b`) stay attached to G's graph.
3. Compute `total_loss_G` = adv + cyc + idt (+ roi_adv, roi_cycle if enabled) -- single sum, one graph.
4. `opt_G.zero_grad()` -> `total_loss_G.backward()` -> clip grad norm -> `opt_G.step()`.
   D's parameters have `requires_grad=False` so no `.grad` accumulates on them, even though the
   backward pass runs through D's forward computation to reach G.

**Step 2 -- Discriminator update** (every `d_update_every` steps)
1. Unfreeze D_A/D_B/roi_disc: `requires_grad_(True)`.
2. Detach fakes: `fake_a = out.x_hat_a.detach()`, `fake_b = out.x_hat_b.detach()` (or replay-buffer
   fakes, also detached) -- stops gradient from reaching G during this update.
3. Image D: `opt_D.zero_grad()` -> `(L_D_A + L_D_B).backward()` (+ R1 penalty every `r1_every`
   steps) -> `opt_D.step()`.
4. ROI D (separate network, separate optimizer, only if `--use_roi_discriminator`):
   `opt_D_roi.zero_grad()` -> `losses_D_roi["total"].backward()` -> `opt_D_roi.step()`.
   Never summed with the image D loss -- different network, different optimizer.

Freeze/unfreeze toggles which parameters *accumulate* gradient; `.detach()` controls whether the
graph *reaches* G at all. Both are needed: G's step relies on D staying differentiable-but-frozen,
D's step relies on the fakes being severed from G's graph.